In [ ]:
### For an overview of scDRS results, we overlay the computed scDRS disease scores on the UMAP.

import os
os.environ.setdefault("MPLCONFIGDIR", "/dski/nobackup/beileib/SVA/.cache/matplotlib")
os.environ.setdefault("NUMBA_CACHE_DIR", "/dski/nobackup/beileib/SVA/.cache/numba")

from pathlib import Path
import scdrs
import scanpy as sc
sc.set_figure_params(dpi=125)
from anndata import AnnData
from scipy import stats
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

In [ ]:
case_dir = Path("/dski/nobackup/beileib/SVA/case_study")
adata_file = case_dir / "Maciej_Daniszewski_2022_case_POAG_noSingletons_with_umap.h5ad"
if not adata_file.exists():
    adata_file = case_dir / "Maciej_Daniszewski_2022_case_POAG_with_umap.h5ad"
adata = sc.read_h5ad(adata_file)

if "Singletons" in set(adata.obs["cell_type"].astype(str)):
    adata = adata[adata.obs["cell_type"].astype(str) != "Singletons"].copy()
    for key in ["X_umap", "X_pca"]:
        if key in adata.obsm:
            del adata.obsm[key]
    for key in ["pca", "neighbors", "umap"]:
        if key in adata.uns:
            del adata.uns[key]

dict_score = {
    "glaucoma": pd.read_csv(
        case_dir / "glaucoma_scdrs/glaucoma.full_score.gz",
        sep="\t",
        index_col=0,
        usecols=lambda col: col in {"Unnamed: 0", "norm_score"},
    )
}

for trait, df_score in dict_score.items():
    adata.obs[trait] = df_score["norm_score"].reindex(adata.obs_names).astype(float)
    print(f"{trait}: {adata.obs[trait].notna().sum()} / {adata.n_obs} cells matched")

if "X_umap" not in adata.obsm:
    sc.pp.highly_variable_genes(adata, n_top_genes=3000, flavor="seurat")
    sc.tl.pca(adata, n_comps=50, use_highly_variable=True, svd_solver="arpack")
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=50)
    sc.tl.umap(adata, random_state=0)

fig_dir = case_dir / "figures"
fig_dir.mkdir(exist_ok=True)

sc.set_figure_params(figsize=[4, 4], dpi=150)
ax = sc.pl.umap(adata, color="cell_type", frameon=False, show=False, size=6)
ax.figure.savefig(fig_dir / "maciej_poag_noSingletons_cluster_umap.png", bbox_inches="tight", dpi=300)
ax.figure.savefig(fig_dir / "maciej_poag_noSingletons_cluster_umap.pdf", bbox_inches="tight")
plt.close(ax.figure)

ax = sc.pl.umap(
    adata,
    color="glaucoma",
    color_map="RdBu_r",
    vmin=-5,
    vmax=5,
    frameon=False,
    show=False,
    size=6,
)
ax.figure.savefig(fig_dir / "maciej_poag_noSingletons_glaucoma_norm_score_umap.png", bbox_inches="tight", dpi=300)
ax.figure.savefig(fig_dir / "maciej_poag_noSingletons_glaucoma_norm_score_umap.pdf", bbox_inches="tight")
plt.close(ax.figure)

adata.write_h5ad(case_dir / "Maciej_Daniszewski_2022_case_POAG_noSingletons_with_umap.h5ad")

In [ ]:
case_dir = Path("/dski/nobackup/beileib/SVA/case_study")
fig_dir = case_dir / "figures"
fig_dir.mkdir(exist_ok=True)

def plot_scdrs_global_heatmap(group_file, out_prefix, trait_label="Glaucoma"):
    df_group = pd.read_csv(group_file, sep="\t")
    df_group["prop_fdr_0.1"] = df_group["n_fdr_0.1"] / df_group["n_cell"]
    df_group["cell_type"] = df_group["group"].astype(str).str.replace("_", " ").str.title()

    heatmap_data = df_group.set_index("cell_type")[["prop_fdr_0.1"]].T
    heatmap_data.index = [trait_label]
    sig_mask = df_group.set_index("cell_type")[["assoc_mcp"]].T < 0.05
    sig_mask.index = heatmap_data.index

    n_rows, n_cols = heatmap_data.shape
    fig_w = max(2.5, 0.28 * n_cols + 0.9)
    fig_h = 1.2
    fig = plt.figure(figsize=(fig_w, fig_h))
    left = 0.24
    bottom = 0.28
    width = 0.7
    cell_w_in = width * fig_w / n_cols
    height = cell_w_in * n_rows / fig_h
    ax = fig.add_axes([left, bottom, width, height])
    im = ax.pcolormesh(
        np.arange(n_cols + 1),
        np.arange(n_rows + 1),
        heatmap_data.values,
        cmap="Reds",
        vmin=0,
        vmax=max(0.1, float(np.nanmax(heatmap_data.values))),
        edgecolors="0.55",
        linewidth=0.8,
        shading="flat",
    )
    ax.set_aspect("equal")
    ax.set_xlim(0, n_cols)
    ax.set_ylim(n_rows, 0)

    ax.set_xticks(np.arange(n_cols) + 0.5)
    ax.set_xticklabels(heatmap_data.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(np.arange(n_rows) + 0.5)
    ax.set_yticklabels(heatmap_data.index, fontsize=8)
    ax.grid(False)
    ax.tick_params(axis="both", length=0)
    ax.set_xlabel("")
    ax.set_ylabel("")

    cax = fig.add_axes([left + 0.18 * width, bottom + height + 0.22, 0.7 * width, 0.06])
    cbar = fig.colorbar(im, cax=cax, orientation="horizontal")
    cbar.set_label("Prop. of sig. cells (FDR < 0.1)", fontsize=9)
    cbar.ax.xaxis.set_label_position("top")
    cbar.ax.xaxis.tick_top()
    cbar.ax.tick_params(labelsize=8)

    for y, trait in enumerate(heatmap_data.index):
        for x, cell_type in enumerate(heatmap_data.columns):
            if bool(sig_mask.loc[trait, cell_type]):
                ax.add_patch(plt.Rectangle((x, y), 1, 1, fill=False, edgecolor="black", lw=1.4))

    fig.savefig(fig_dir / f"{out_prefix}.png", bbox_inches="tight", dpi=300)
    fig.savefig(fig_dir / f"{out_prefix}.pdf", bbox_inches="tight")
    plt.close(fig)


plot_scdrs_global_heatmap(
    case_dir / "glaucoma_scdrs/glaucoma.scdrs_group.cell_type",
    "glaucoma_cell_type_global_heatmap",
)
plot_scdrs_global_heatmap(
    case_dir / "glaucoma_scdrs1/glaucoma.scdrs_group.cell_type",
    "maciej_daniszewski_2022organoid_cell_type_global_heatmap",
)